# img2text worker — Kaggle GPU runner

Cell order per brief: Setup → Secrets → Smoke (ONCE, manually, before the loop) → Init → Loop.

The Smoke cell loads the pinned pipeline itself (runs before Init) and prints the ACTUAL
`res.json` keys — record them in `docs/colab-smoke-checklist.md`. Init reuses that
pipeline instance when present (no double model load).

In [ ]:
# Setup
!apt-get -qq install -y poppler-utils
!pip install -q paddlepaddle-gpu paddleocr httpx opencv-python-headless rapidfuzz pypdf pillow numpy
!pdftoppm -v

In [ ]:
# Secrets — retry until both are set
import time
from kaggle_secrets import UserSecretsClient

SECRETS = UserSecretsClient()

while True:
    try:
        BASE_URL = SECRETS.get_secret("BASE_URL")
        RUNNER_KEY = SECRETS.get_secret("RUNNER_KEY")
        break
    except Exception:
        print("Set BASE_URL and RUNNER_KEY in Kaggle Secrets (Add-ons → Secrets)")
        time.sleep(5)

print("BASE_URL:", BASE_URL)
print("RUNNER_KEY: set (", len(RUNNER_KEY), "chars )")


## Smoke — run ONCE manually before the loop

Upload an image as `TA.jpg` into `/content/`, then run this cell once.
Compare the printed `res.json` keys against the adapter assumption in `worker/pipeline.py`
(layout list with `type`/`bbox`/`rec_texts`/`rec_scores` + `markdown`).
Record the ACTUAL keys in `docs/colab-smoke-checklist.md`.

In [ ]:
# Smoke — ONCE, manually, before the loop (self-contained: loads pinned pipeline)
import pathlib

from paddleocr import PaddleOCRVL

PIPELINE = PaddleOCRVL(
    pipeline_version="v1.5",
    vl_rec_model_name="PaddleOCR-VL-1.5-0.9B",
    layout_detection_model_name="PP-DocLayoutV3",
)
res = PIPELINE.predict("TA.jpg")
print("res.json keys:", list(res.json.keys()))

out = pathlib.Path("/tmp/smoke_out")
out.mkdir(parents=True, exist_ok=True)
for key, value in res.json.items():
    path = out / f"{key}.txt"
    path.write_text(str(value), encoding="utf-8")
    print("saved", path)
if "markdown" in res.json:
    (out / "page.md").write_text(res.json["markdown"], encoding="utf-8")
    print("saved /tmp/smoke_out/page.md")

In [ ]:
# Init — pinned API from spec §7; reuses PIPELINE from the smoke cell when present
from paddleocr import PaddleOCRVL

if "PIPELINE" not in globals():
    PIPELINE = PaddleOCRVL(
        pipeline_version="v1.5",
        vl_rec_model_name="PaddleOCR-VL-1.5-0.9B",
        layout_detection_model_name="PP-DocLayoutV3",
    )
print("pipeline ready:", type(PIPELINE).__name__)

In [ ]:
# Loop — after the smoke cell passes
import sys
import time

import httpx

sys.path.insert(0, "/kaggle/input/img2text-worker-files")
from pipeline import run_job

jobs = run_job(
    BASE_URL,
    RUNNER_KEY,
    httpx.Client(timeout=60),
    pipeline_factory=lambda: PIPELINE,
    sleep_fn=time.sleep,
)
print("loop stopped after", jobs, "jobs")